In [1]:
import torch
import torch.nn as nn
from einops import rearrange

In [2]:
class PhysicsAttentionIrregularMesh(nn.Module):
    """
    This module implements the PhysicsAttention mechanism for irregular meshes 
        from Transolver (https://arxiv.org/pdf/2402.02366). The code is adapted from 
        https://github.com/thuml/Transolver/blob/main/Physics_Attention.py For 
        regular meshes, Transolver uses Conv2d layers for the `in_project*` layers.

        
    Inputs are expected in the shape (B, N, C) where B is batch size, N 
        is the number of (irregular) nodes, and C is the node feature dimension.
    """

    def __init__(self, dim, num_attn_heads=8, attn_head_dim=64, dropout=0., slice_num=64):
        super().__init__()
        self.attn_head_dim = attn_head_dim
        self.num_attn_heads = num_attn_heads
        self.inner_dim = self.attn_head_dim * self.num_attn_heads

        self.scale = self.attn_head_dim ** -0.5

        self.dropout = nn.Dropout(dropout)

        self.softmax = nn.Softmax(dim=-1)
        self.softmax_temperature_scaling = nn.Parameter(torch.ones([1, self.num_attn_heads, 1, 1]) * 0.5)

        self.in_project_x = nn.Linear(dim, self.inner_dim)
        self.in_project_fx = nn.Linear(dim, self.inner_dim)
        self.in_project_slice = nn.Linear(self.attn_head_dim, slice_num)
        for l in [self.in_project_slice]:
            torch.nn.init.orthogonal_(l.weight)  # use a principled initialization
        self.to_q = nn.Linear(self.attn_head_dim, self.attn_head_dim, bias=False)
        self.to_k = nn.Linear(self.attn_head_dim, self.attn_head_dim, bias=False)
        self.to_v = nn.Linear(self.attn_head_dim, self.attn_head_dim, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(self.inner_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # B N C
        B, N, C = x.shape

        ## 1 SLICE
        fx_mid = self.in_project_fx(x).reshape(B, N, self.num_attn_heads, self.attn_head_dim) \
            .permute(0, 2, 1, 3).contiguous()  # B H N C
        x_mid = self.in_project_x(x).reshape(B, N, self.num_attn_heads, self.attn_head_dim) \
            .permute(0, 2, 1, 3).contiguous()  # B H N C
        slice_weights = self.softmax(self.in_project_slice(x_mid) / self.softmax_temperature_scaling)  # B H N G
        slice_norm = slice_weights.sum(2)  # B H G
        slice_token = torch.einsum("bhnc,bhng->bhgc", fx_mid, slice_weights)
        slice_token = slice_token / ((slice_norm + 1e-5)[:, :, :, None].repeat(1, 1, 1, self.attn_head_dim))

        ## 2 ATTENTION
        q_slice_token = self.to_q(slice_token)
        k_slice_token = self.to_k(slice_token)
        v_slice_token = self.to_v(slice_token)
        dots = torch.matmul(q_slice_token, k_slice_token.transpose(-1, -2)) * self.scale
        attn = self.softmax(dots)
        attn = self.dropout(attn)
        out_slice_token = torch.matmul(attn, v_slice_token)  # B H G D

        # 3 DESLICE
        out_x = torch.einsum("bhgc,bhng->bhnc", out_slice_token, slice_weights)
        out_x = rearrange(out_x, 'b h n d -> b n (h d)')
        return self.to_out(out_x)